In [ ]:
import os
import getpass
import numpy as np
from langchain_docling.loader import DoclingLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:
if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

In [ ]:
FILE_PATH = r"docs\Linix_Commands_Cheatsheet.pdf"

## LOADERS

In [ ]:
loader = DoclingLoader(FILE_PATH)

In [ ]:
# Load all documents into the memory immediately
documents = loader.load()   # returns a list of Document objects.


# # For large datasets, lazily load documents
# # it is used when pdf is too large and cannot fit-in memory at once  
# for document in loader.lazy_load():   # Returns a generator that yields one document at a time.
#     print(document)

In [ ]:
# every Document obj has two attributes, metadata and page_content
for index, document in enumerate(documents):
    print(f"\n---- Document {index+1} ----")
    print(document.page_content)

In [ ]:
# See which document contains which page of pdf
for i, document in enumerate(documents):
    print(f"\nDocument {i}")

    pages = []

    for item in document.metadata["dl_meta"]["doc_items"]:
        for prov in item["prov"]:
            pages.append(prov["page_no"])

    print("Pages:", sorted(set(pages)))

## SPLITTERS

In [ ]:
# first convert the all documents into string as we have to pass it to text splitter (which takes atring)
full_text = "\n".join([doc.page_content for doc in documents])

In [ ]:
print(full_text)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
texts = text_splitter.split_text(full_text)     # # it create chunks
print(texts)

## EMBEDDINGS

In [ ]:
# select embeddings model
# embeddings is object
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)

In [ ]:
# first make chunks of make embeddings of each document
document_embeddings = embeddings.embed_documents(texts)

In [ ]:
query = "What is the use of git cd command?"
query_embedding = embeddings.embed_query(query)

In [ ]:
for index, doc_embedding in enumerate(document_embeddings):
    print(f"\nDocument Embeddings {index+1}")
    print(doc_embedding)

print(f"\nQuery Embeddings\n{query_embedding}")

In [ ]:
# defining our our own function to find similarities, if we use vector store, that provide this built-in 
def cosine_similarities(vect1, vect2):
    dot_product = np.dot(vect1, vect2)
    norm1 = np.linalg.norm(vect1)
    norm2 = np.linalg.norm(vect2)
    return dot_product / (norm1 * norm2)

In [ ]:
cosine_similarities(document_embeddings, query_embedding)

## VECTOR STORE

In [ ]:
vector_store = InMemoryVectorStore(embedding=embeddings)      # create vector store in RAM not in hard disc (for learning)
# we can use diff vector store / db for permanent storage like chroma, pinecore, FIASS

In [ ]:
ids = vector_store.add_documents(documents=documents)
print(ids)

In [ ]:
# view documents in vector store
doc = vector_store.get_by_ids( ['bb7538b8-b792-4000-b01c-7a943c440c8a'] )
print((doc))

In [ ]:
docs = vector_store.similarity_search(
    query=query,
    k=1     # how many top matching documents you want
)

In [ ]:
for doc in docs:
    print('id:', doc.id)
    print(doc.page_content)
    print()


In [ ]:
docs = vector_store.similarity_search_with_score(   # also give the score
    query=query,
    k=1,     # how many top matching documents you want
    filter=lambda doc: doc.metadata.get("source") == "docs\\Linix_Commands_Cheatsheet.pdf"   # conditional filtering based on metadata, filter os OnMemoryVectorStore accept callable obj while many other vector dbs accepts dict
)

In [ ]:
for doc in docs:
    print(doc[-1])      # similarity score, less score more similarity (less score means less angular distancs)

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)